# 27 Local Baseline ML Modeling
Local-only baseline modeling from Phase 26 features.


**Notebook purpose:** Trains softmax regression and bagged stump ensemble baselines on engagement features. Produces confusion matrices, feature importance, and prediction samples.

**Required data:** `local/derived/features/bluesky_engagement_features.parquet` (from notebook 26).

**Run order:** Run after notebook 26 (feature engineering). Run before notebook 28 (refinement/ablation).

## 1) Load Data and Inspect Actual Schema


In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'modeling' / 'baseline_model.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/modeling/baseline_model.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.baseline_model import (
    ModelingConfig,
    build_column_groups,
    run_baseline_modeling,
    write_modeling_artifacts,
)

feature_path = ROOT / 'local/derived/features/bluesky_engagement_features.parquet'
feature_summary_path = ROOT / 'local/derived/features/bluesky_engagement_feature_summary.json'

if not feature_path.exists():
    raise FileNotFoundError(
        "DATA NOT YET AVAILABLE -- run notebook 26 (feature engineering) first.\n"
        f"Missing: {feature_path}"
    )

feature_df = pd.read_parquet(feature_path)
feature_summary = None
if feature_summary_path.exists():
    payload = json.loads(feature_summary_path.read_text())
    feature_summary = payload.get('summary')

print('rows:', len(feature_df), 'cols:', len(feature_df.columns))
print('target distribution:')
print(feature_df['engagement_label'].astype(str).value_counts())

rows: 19999 cols: 92
target distribution:
engagement_label
LOW       17124
MEDIUM     1568
HIGH       1307
Name: count, dtype: int64


In [3]:
print('columns:')
print(feature_df.columns.tolist())


columns:
['uri', 'post_created_at', 'source_run_tag', 'text_source', 'raw_capture_run_id', 'raw_captured_at', 'raw_repo_did', 'raw_record_created_at', 'hydrated_capture_run_id', 'hydrated_hydrate_run_id', 'hydrated_author_did', 'hydrated_author_handle', 'hydrated_indexed_at', 'hydrated_hydrated_at', 'raw_source_row_json', 'hydrated_source_row_json', 'post_text_raw', 'post_text_clean', 'post_text_alnum', 'post_token_count', 'post_char_count', 'has_hashtag', 'has_url', 'has_mention', 'has_special_chars', 'has_non_ascii', 'match_stage', 'match_score', 'match_method', 'temporal_pool_mode', 'trend_name_clean', 'trend_date', 'trend_counts', 'trend_num_hours', 'is_matched', 'is_ambiguous', 'candidate_count', 'matched_candidate_count', 'matched_candidate_rate', 'eng_author_did', 'eng_author_handle', 'like_count', 'reply_count', 'repost_count', 'quote_count', 'indexed_at', 'hydrated_at', 'capture_run_id', 'hydrate_run_id', 'actor_did', 'actor_handle', 'followers_count', 'follows_count', 'posts_

## 2) Define Target / Model / Debug Columns and Exclusions


In [4]:
groups = build_column_groups(feature_df, feature_summary=feature_summary)
print('target:', groups['target_column'])
print('model feature count:', len(groups['model_feature_columns']))
print('debug column count:', len(groups['debug_columns']))
print('label columns:', groups['label_columns'])
print('excluded columns:', groups['excluded_columns'])


target: engagement_label
model feature count: 44
debug column count: 19
label columns: ['engagement_label', 'engagement_total', 'eng_like_count', 'eng_reply_count', 'eng_repost_count', 'eng_quote_count']
excluded columns: {'leakage': ['eng_like_count', 'eng_quote_count', 'eng_reply_count', 'eng_repost_count', 'engagement_label', 'engagement_total', 'like_count', 'quote_count', 'reply_count', 'repost_count'], 'debug_reference': ['actor_did', 'actor_handle', 'eng_author_did', 'eng_author_handle', 'hydrated_author_did', 'hydrated_author_handle', 'hydrated_source_row_json', 'match_method', 'post_created_at', 'post_text_alnum', 'post_text_clean', 'post_text_raw', 'raw_source_row_json', 'source_run_tag', 'temporal_pool_mode', 'text_source', 'trend_date', 'trend_name_clean', 'uri'], 'high_cardinality': [], 'unsupported_dtype': ['actor_created_at', 'actor_indexed_at', 'actor_run_id', 'capture_run_id', 'description', 'display_name', 'hydrate_run_id', 'hydrated_at', 'hydrated_capture_run_id', 'h

## 3) Train and Evaluate Baselines


In [5]:
config = ModelingConfig(
    random_seed=42,
    test_size=0.30,
    logistic_learning_rate=0.10,
    logistic_max_iter=1200,
    logistic_l2=0.001,
    stump_estimators=200,
    stump_max_features=None,
    stump_threshold_count=9,
)

modeling_output = run_baseline_modeling(
    feature_df=feature_df,
    feature_summary=feature_summary,
    config=config,
)

results = modeling_output['results']
print('split:', results['split'])
print('best_model:', results['best_model_name'])
print('criterion:', results['best_model_criterion'])


split: {'train_size': 14000, 'test_size': 5999, 'test_size_ratio': 0.2999649982499125, 'train_class_distribution': {'HIGH': 915, 'LOW': 11987, 'MEDIUM': 1098}, 'test_class_distribution': {'HIGH': 392, 'LOW': 5137, 'MEDIUM': 470}}
best_model: bagged_stump_ensemble
criterion: macro_f1_then_balanced_accuracy_then_accuracy


In [6]:
softmax_metrics = results['models']['softmax_regression']['metrics']
stump_metrics = results['models']['bagged_stump_ensemble']['metrics']
print('softmax metrics:', softmax_metrics)
print()
print('stump metrics:', stump_metrics)


softmax metrics: {'accuracy': 0.4072345390898483, 'balanced_accuracy': 0.4167678753126837, 'macro_precision': 0.3677890361112568, 'macro_recall': 0.4167678753126837, 'macro_f1': 0.29266047646782845, 'per_class': {'HIGH': {'precision': 0.09120521172638436, 'recall': 0.42857142857142855, 'f1': 0.1504028648164727, 'support': 392}, 'LOW': {'precision': 0.9070680628272252, 'recall': 0.40471092077087795, 'f1': 0.5596984789339077, 'support': 5137}, 'MEDIUM': {'precision': 0.10509383378016086, 'recall': 0.41702127659574467, 'f1': 0.16788008565310494, 'support': 470}}, 'confusion_matrix': [[168, 91, 133], [1522, 2079, 1536], [152, 122, 196]]}

stump metrics: {'accuracy': 0.8563093848974829, 'balanced_accuracy': 0.3333333333333333, 'macro_precision': 0.2854364616324943, 'macro_recall': 0.3333333333333333, 'macro_f1': 0.3075311302681992, 'per_class': {'HIGH': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 392}, 'LOW': {'precision': 0.8563093848974829, 'recall': 1.0, 'f1': 0.9225933908045

## 4) Confusion Matrix and Error Analysis


In [7]:
confusion_df = results['confusion_matrix']
confusion_df


,HIGH,LOW,MEDIUM
HIGH,0,392,0
LOW,0,5137,0
MEDIUM,0,470,0


In [8]:
pred_df = results['prediction_sample']
print('prediction sample rows:', len(pred_df))
print('incorrect predictions:', int((~pred_df['is_correct']).sum()))
pred_df[['uri','true_label','predicted_label','best_model_name','is_correct']].head(20)


prediction sample rows: 5999
incorrect predictions: 862


,uri,true_label,predicted_label,best_model_name,is_correct
0,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
1,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
2,at://did:plc:22ezkuvas6f545oal47snp5x/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
3,at://did:plc:22hp52c2fr5ywmvqhfomzqja/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
4,at://did:plc:22k3roaumaq77xwhhpdcjdka/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
5,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
6,at://did:plc:22ximgct4dxbmv4hlekxmnmp/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
7,at://did:plc:23mfy2ig3bttvd6hrtxi64wv/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
8,at://did:plc:23mfy2ig3bttvd6hrtxi64wv/app.bsky...,LOW,LOW,bagged_stump_ensemble,True
9,at://did:plc:244dhc5kw7asrhd7i4li7khx/app.bsky...,LOW,LOW,bagged_stump_ensemble,True


In [9]:
if 'true_label' in pred_df.columns and 'predicted_label' in pred_df.columns:
    confusion_pairs = (
        pred_df.loc[pred_df['true_label'] != pred_df['predicted_label'], ['true_label', 'predicted_label']]
        .value_counts()
    )
    print('common confusion pairs:')
    print(confusion_pairs if len(confusion_pairs) else 'none')


common confusion pairs:
true_label  predicted_label
MEDIUM      LOW                470
HIGH        LOW                392
Name: count, dtype: int64


## 5) Feature Importance / Interpretability


In [10]:
importance_df = results['feature_importance']
print('importance rows:', len(importance_df))
importance_df.head(20)


importance rows: 280


,model_name,feature_name,importance,class_label,signed_value
0,bagged_stump_ensemble,num__followers_count,0.245204,__all__,0.245204
1,bagged_stump_ensemble,num__follows_count,0.153042,__all__,0.153042
2,bagged_stump_ensemble,num__actor_follows_count,0.148567,__all__,0.148567
3,bagged_stump_ensemble,num__actor_followers_count,0.133573,__all__,0.133573
4,bagged_stump_ensemble,cat__has_banner==False,0.072079,__all__,0.072079
5,bagged_stump_ensemble,cat__has_banner==True,0.071716,__all__,0.071716
6,bagged_stump_ensemble,num__actor_has_banner,0.047144,__all__,0.047144
7,bagged_stump_ensemble,num__actor_followers_to_follows_ratio,0.031289,__all__,0.031289
8,bagged_stump_ensemble,num__actor_description_char_count,0.018465,__all__,0.018465
9,bagged_stump_ensemble,num__posts_count,0.015058,__all__,0.015058


In [11]:
print('top softmax (__all__) features:')
print(
    importance_df.loc[
        (importance_df['model_name'] == 'softmax_regression')
        & (importance_df['class_label'] == '__all__')
    ]
    .head(15)
    [['feature_name','importance','signed_value']]
    .to_string(index=False)
)
print()
print('top stump features:')
print(
    importance_df.loc[importance_df['model_name'] == 'bagged_stump_ensemble']
    .head(15)
    [['feature_name','importance']]
    .to_string(index=False)
)


top softmax (__all__) features:
                feature_name  importance  signed_value
    num__actor_profile_found    0.098227     -0.001527
 num__actor_has_display_name    0.093586     -0.004680
  num__actor_followers_count    0.090118      0.003118
num__post_unique_token_count    0.088099      0.000923
        num__candidate_count    0.086964      0.003657
        num__followers_count    0.084278     -0.004305
          num__post_hour_utc    0.083844     -0.006254
       num__actor_has_banner    0.082085     -0.003728
      cat__has_banner==False    0.078951     -0.004308
       num__actor_has_avatar    0.075211     -0.010041
     cat__match_stage==fuzzy    0.074994      0.003395
        num__post_char_count    0.073731     -0.003225
                num__has_url    0.073366      0.002949
      num__actor_posts_count    0.069486     -0.004684
            num__posts_count    0.067707     -0.006455

top stump features:
                         feature_name  importance
                 

## 6) Write Phase 27 Artifacts


In [12]:
artifact_paths = write_modeling_artifacts(
    modeling_output=modeling_output,
    output_dir=str(ROOT / 'local/derived/modeling'),
    sample_csv_path=str(ROOT / 'data/samples/baseline_predictions_sample_1000.csv'),
)
artifact_paths

{'metrics_json': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/baseline_model_metrics.json',
 'feature_importance_parquet': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/baseline_feature_importance.parquet',
 'prediction_sample_parquet': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/baseline_predictions_sample.parquet',
 'prediction_sample_csv': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/baseline_predictions_sample_1000.csv',
 'summary_json': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/baseline_model_summary.json',
 'confusion_matrix_csv': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass

## 7) Readiness Statement
Phase 27 is complete when metrics, confusion matrix, feature importance, and prediction sample artifacts are written and documented.
